# 🚀 Weave GGUF Model Server
### Free T4 GPU Hosting for `darthnihilus01/weave-llama3.2-3b` via Cloudflare Tunnel

Run these 3 simple cells in order:

### Step 1: Install `zstd`, Ollama with GPU support, and Cloudflare Tunnel

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -L https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst | tar --zstd -xf - -C /usr/local
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!ollama --version

### Step 2: Start Ollama & Pull Your Model from Hugging Face

In [ ]:
import os, subprocess, time

# Stop any previous instance
!pkill -9 ollama || true

# Set OLLAMA_ORIGINS='*' so Ollama allows Cloudflare Tunnel proxy traffic
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

# Start Ollama GPU daemon in background
subprocess.Popen(["/usr/local/bin/ollama", "serve"])
time.sleep(3)

# Pull your GGUF model directly from Hugging Face with live progress bar
!ollama pull hf.co/darthnihilus01/weave-llama3.2-3b

### Step 3: Start Cloudflare Tunnel & Get Your Live URL

In [ ]:
import subprocess, time, re

print("🌐 Starting Cloudflare Tunnel...")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for line in iter(cf_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "="*65)
    print("🎉 YOUR WEAVE CUSTOM GGUF MODEL IS LIVE!")
    print("="*65)
    print("\nUpdate your Weave .env file with these lines:\n")
    print(f"ROUTER_BASE_URL={tunnel_url}/v1")
    print("ROUTER_API_KEY=colab")
    print("ROUTER_MODEL=hf.co/darthnihilus01/weave-llama3.2-3b")
    print("\n" + "="*65)
    print("Keep this Colab tab running while using Weave!\n")
else:
    print("❌ Failed to obtain tunnel URL.")

# Keep cell alive
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Server stopped.")